<a href="https://colab.research.google.com/github/Mageed-Ghaleb/OptimizationSystems-Course/blob/main/Lab_05.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lab 5 — Metaheuristic Design on a Single-Machine Scheduling Case Study (Colab-ready)

**Goal.** Design and compare several metaheuristics on the *same* combinatorial optimization problem so you can see how:
- neighborhood-based methods (Local Search, Simulated Annealing, Tabu Search),
- population-based methods (Genetic Algorithm),
- swarm/constructive methods adapted to permutations (PSO via random keys, Ant Colony Optimization)

behave across iterations and parameters.

We use **single-machine scheduling** with a permutation decision (the job sequence).  
You can easily **add more jobs** by editing the data cell (or by generating a random instance).

## 0) Setup (install + imports)

Colab already includes most packages below, but we keep this cell for portability.

In [ ]:
# Install (safe to re-run)
!pip -q install numpy matplotlib pandas

import numpy as np
import pandas as pd
import math, random, time
import matplotlib.pyplot as plt

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

## 1) Problem definition

**Decision variable:** a permutation (ordering) of jobs, e.g. `[3, 1, 0, 2, ...]`.

Each job `j` has:
- processing time `p[j]`
- due date `d[j]`

For a given sequence, completion times are:
- \( C_{k} = \sum_{t \le k} p[\text{seq}[t]] \)

Tardiness of job `j` in the sequence is:
- \( T_j = \max(0, C_j - d_j) \)

**Objective:** minimize total tardiness  
\[
\min \sum_j T_j
\]

This objective is simple to compute by hand (good for learning) and still nontrivial for heuristics.

## 2) Data cell (edit here to add more jobs)

You can:
- **Manually edit** `jobs_df`, or
- **Generate** a random instance with `make_random_instance(n_jobs=...)`.

Everything below reads from `jobs_df`, so scaling up is as simple as changing this one cell.

In [ ]:
# Option A: manual instance (edit freely)
jobs_df = pd.DataFrame({
    "job": list(range(10)),
    "p":  [3, 2, 7, 5, 4, 6, 2, 8, 3, 5],
    "d":  [4, 7, 12, 10, 9, 15, 6, 18, 11, 14],
})
jobs_df

In [ ]:
# Option B: random instance generator (uncomment to use)
def make_random_instance(n_jobs=12, p_low=1, p_high=10, due_factor=1.3, seed=42):
    rng = np.random.default_rng(seed)
    p = rng.integers(p_low, p_high+1, size=n_jobs)
    avg_p = float(np.mean(p))
    d = rng.integers(int(avg_p*0.8), int(avg_p*due_factor*n_jobs/3)+1, size=n_jobs)
    return pd.DataFrame({"job": list(range(n_jobs)), "p": p.tolist(), "d": d.tolist()})

# jobs_df = make_random_instance(n_jobs=14, seed=SEED)
# jobs_df

## 3) Core utilities (objective, decoding, neighborhoods)

We keep these functions small and reusable across all algorithms.

In [ ]:
def total_tardiness(seq, p, d):
    C = 0
    tt = 0
    for j in seq:
        C += p[j]
        tt += max(0, C - d[j])
    return tt

def schedule_table(seq, p, d):
    rows = []
    C = 0
    for pos, j in enumerate(seq):
        C += p[j]
        T = max(0, C - d[j])
        rows.append({"pos": pos, "job": j, "p": p[j], "d": d[j], "C": C, "T": T})
    return pd.DataFrame(rows)

def swap_move(seq, i, k):
    s = list(seq)
    s[i], s[k] = s[k], s[i]
    return s

def insert_move(seq, i, k):
    s = list(seq)
    job = s.pop(i)
    s.insert(k, job)
    return s

def neighbors_swap(seq):
    n = len(seq)
    for i in range(n):
        for k in range(i+1, n):
            yield (("swap", i, k), swap_move(seq, i, k))

def neighbors_insert(seq):
    n = len(seq)
    for i in range(n):
        for k in range(n):
            if i == k:
                continue
            yield (("ins", i, k), insert_move(seq, i, k))

def random_neighbor(seq, move_type="swap"):
    n = len(seq)
    if move_type == "swap":
        i, k = random.sample(range(n), 2)
        if i > k:
            i, k = k, i
        return ("swap", i, k), swap_move(seq, i, k)
    elif move_type == "insert":
        i = random.randrange(n)
        k = random.randrange(n)
        while k == i:
            k = random.randrange(n)
        return ("ins", i, k), insert_move(seq, i, k)
    else:
        raise ValueError("move_type must be 'swap' or 'insert'")

p = jobs_df.set_index("job")["p"].to_dict()
d = jobs_df.set_index("job")["d"].to_dict()

n_jobs = len(jobs_df)
all_jobs = list(range(n_jobs))

edd_seq = sorted(all_jobs, key=lambda j: d[j])
edd_obj = total_tardiness(edd_seq, p, d)

edd_obj, edd_seq

In [ ]:
schedule_table(edd_seq, p, d)

## 4) Local Search (LS)

Best-improvement local search with swap neighborhood. Stop when no improvement exists.

In [ ]:
def local_search_best_improvement(init_seq, p, d, neighborhood="swap", max_iters=2000):
    seq = list(init_seq)
    best = total_tardiness(seq, p, d)
    history = [best]
    it = 0

    while it < max_iters:
        it += 1
        improved = False
        best_candidate = None
        best_val = best

        gen = neighbors_swap(seq) if neighborhood == "swap" else neighbors_insert(seq)

        for _, cand in gen:
            val = total_tardiness(cand, p, d)
            if val < best_val:
                best_val = val
                best_candidate = cand
                improved = True

        if not improved:
            break

        seq = best_candidate
        best = best_val
        history.append(best)

    return seq, best, history

In [ ]:
ls_seq, ls_obj, ls_hist = local_search_best_improvement(edd_seq, p, d, neighborhood="swap")
ls_obj, len(ls_hist)

## 5) Simulated Annealing (SA)

Accept worse moves with probability exp(-delta / T). Temperature cools geometrically.

In [ ]:
def simulated_annealing(init_seq, p, d, n_iters=3000, T0=50.0, alpha=0.995, move_type="swap"):
    curr = list(init_seq)
    curr_val = total_tardiness(curr, p, d)
    best = curr[:]
    best_val = curr_val

    T = float(T0)
    best_so_far = [best_val]

    for _ in range(n_iters):
        _, neigh = random_neighbor(curr, move_type=move_type)
        neigh_val = total_tardiness(neigh, p, d)
        delta = neigh_val - curr_val

        accept = (delta <= 0) or (random.random() < math.exp(-delta / max(T, 1e-12)))
        if accept:
            curr, curr_val = neigh, neigh_val
            if curr_val < best_val:
                best, best_val = curr[:], curr_val

        T *= alpha
        best_so_far.append(best_val)

    return best, best_val, best_so_far

In [ ]:
sa_seq, sa_obj, sa_hist = simulated_annealing(edd_seq, p, d, n_iters=5000, T0=60.0, alpha=0.997, move_type="swap")
sa_obj, len(sa_hist)

## 6) Tabu Search (TS)

Swap neighborhood, tabu attribute is swapped job-pair, fixed tenure, aspiration if it beats global best.

In [ ]:
def tabu_search(init_seq, p, d, n_iters=2000, tenure=10):
    curr = list(init_seq)
    curr_val = total_tardiness(curr, p, d)
    best = curr[:]
    best_val = curr_val

    tabu = {}
    best_so_far = [best_val]

    for _ in range(n_iters):
        # decrement
        for k in list(tabu.keys()):
            tabu[k] -= 1
            if tabu[k] <= 0:
                tabu.pop(k, None)

        best_cand = None
        best_cand_val = float("inf")
        best_attr = None

        for move_attr, cand in neighbors_swap(curr):
            _, i, k = move_attr
            a, b = curr[i], curr[k]
            attr = tuple(sorted((a, b)))
            val = total_tardiness(cand, p, d)

            is_tabu = attr in tabu
            aspirate = val < best_val

            if (not is_tabu) or aspirate:
                if val < best_cand_val:
                    best_cand = cand
                    best_cand_val = val
                    best_attr = attr

        if best_cand is None:
            break

        curr, curr_val = best_cand, best_cand_val
        tabu[best_attr] = tenure

        if curr_val < best_val:
            best, best_val = curr[:], curr_val

        best_so_far.append(best_val)

    return best, best_val, best_so_far

In [ ]:
ts_seq, ts_obj, ts_hist = tabu_search(edd_seq, p, d, n_iters=2500, tenure=12)
ts_obj, len(ts_hist)

## 7) Genetic Algorithm (GA) for permutations

Tournament selection, OX crossover, swap mutation, elitism.

In [ ]:
def order_crossover(p1, p2):
    n = len(p1)
    a, b = sorted(random.sample(range(n), 2))
    child = [None] * n
    child[a:b+1] = p1[a:b+1]
    fill = [x for x in p2 if x not in child]
    idx = 0
    for i in range(n):
        if child[i] is None:
            child[i] = fill[idx]
            idx += 1
    return child

def swap_mutation(seq, pm=0.2):
    s = list(seq)
    if random.random() < pm:
        i, k = random.sample(range(len(s)), 2)
        s[i], s[k] = s[k], s[i]
    return s

def tournament_select(pop, fit, k=3):
    idxs = random.sample(range(len(pop)), k)
    best = min(idxs, key=lambda i: fit[i])
    return pop[best]

def genetic_algorithm(p, d, n_jobs, pop_size=30, n_gens=200, pc=0.9, pm=0.2, elite=2):
    base = list(range(n_jobs))
    pop = []
    for _ in range(pop_size):
        s = base[:]
        random.shuffle(s)
        pop.append(s)

    def eval_pop(pop_):
        return [total_tardiness(ind, p, d) for ind in pop_]

    fit = eval_pop(pop)
    best_idx = int(np.argmin(fit))
    best = pop[best_idx][:]
    best_val = fit[best_idx]
    hist = [best_val]

    for _ in range(n_gens):
        elite_idxs = np.argsort(fit)[:elite]
        new_pop = [pop[i][:] for i in elite_idxs]

        while len(new_pop) < pop_size:
            p1 = tournament_select(pop, fit, k=3)
            p2 = tournament_select(pop, fit, k=3)

            if random.random() < pc:
                c1 = order_crossover(p1, p2)
                c2 = order_crossover(p2, p1)
            else:
                c1, c2 = p1[:], p2[:]

            c1 = swap_mutation(c1, pm=pm)
            c2 = swap_mutation(c2, pm=pm)
            new_pop.append(c1)
            if len(new_pop) < pop_size:
                new_pop.append(c2)

        pop = new_pop
        fit = eval_pop(pop)
        gen_best_idx = int(np.argmin(fit))
        if fit[gen_best_idx] < best_val:
            best_val = fit[gen_best_idx]
            best = pop[gen_best_idx][:]

        hist.append(best_val)

    return best, best_val, hist

In [ ]:
ga_seq, ga_obj, ga_hist = genetic_algorithm(p, d, n_jobs, pop_size=40, n_gens=250, pc=0.9, pm=0.25, elite=2)
ga_obj, len(ga_hist)

## 8) PSO adapted to permutations via random keys

Particles move in continuous key space; permutations are obtained by sorting keys.

In [ ]:
def decode_random_keys(keys):
    return list(np.argsort(keys))

def pso_random_keys(p, d, n_jobs, n_particles=25, n_iters=250, w=0.7, c1=1.4, c2=1.4, vmax=0.5, seed=42):
    rng = np.random.default_rng(seed)
    X = rng.uniform(0, 1, size=(n_particles, n_jobs))
    V = rng.uniform(-0.1, 0.1, size=(n_particles, n_jobs))

    def eval_keys(keys):
        perm = decode_random_keys(keys)
        return total_tardiness(perm, p, d), perm

    pbest_X = X.copy()
    pbest_val = np.zeros(n_particles)
    pbest_perm = []
    for i in range(n_particles):
        val, perm = eval_keys(X[i])
        pbest_val[i] = val
        pbest_perm.append(perm)

    g_idx = int(np.argmin(pbest_val))
    gbest_X = pbest_X[g_idx].copy()
    gbest_val = float(pbest_val[g_idx])
    gbest_perm = pbest_perm[g_idx][:]

    hist = [gbest_val]

    for _ in range(n_iters):
        r1 = rng.uniform(0, 1, size=(n_particles, n_jobs))
        r2 = rng.uniform(0, 1, size=(n_particles, n_jobs))
        V = w * V + c1 * r1 * (pbest_X - X) + c2 * r2 * (gbest_X - X)
        V = np.clip(V, -vmax, vmax)
        X = X + V

        for i in range(n_particles):
            val, perm = eval_keys(X[i])
            if val < pbest_val[i]:
                pbest_val[i] = val
                pbest_X[i] = X[i].copy()
                pbest_perm[i] = perm
                if val < gbest_val:
                    gbest_val = float(val)
                    gbest_X = X[i].copy()
                    gbest_perm = perm[:]

        hist.append(gbest_val)

    return gbest_perm, gbest_val, hist

In [ ]:
pso_seq, pso_obj, pso_hist = pso_random_keys(p, d, n_jobs, n_particles=30, n_iters=300, w=0.7, c1=1.6, c2=1.6, vmax=0.35, seed=SEED)
pso_obj, len(pso_hist)

## 9) Ant Colony Optimization (ACO) for permutations (simple constructive version)

We use pheromone on transitions between jobs plus a simple heuristic based on due date and processing time.

In [ ]:
def aco_scheduling(p, d, n_jobs, n_ants=20, n_iters=120, alpha=1.0, beta=2.0, rho=0.2, q=1.0, seed=42):
    rng = np.random.default_rng(seed)
    jobs = list(range(n_jobs))
    start = n_jobs
    tau = np.ones((n_jobs + 1, n_jobs))

    eta_job = np.array([1.0 / (1.0 + d[j]) * 1.0 / (1.0 + p[j]) for j in jobs])

    def construct_solution():
        remaining = set(jobs)
        seq = []
        prev = start
        while remaining:
            rem_list = list(remaining)
            pher = np.array([tau[prev, j] for j in rem_list]) ** alpha
            heur = np.array([eta_job[j] for j in rem_list]) ** beta
            probs = pher * heur
            s = probs.sum()
            probs = probs / s if s > 0 else np.ones_like(probs) / len(probs)
            j = int(rng.choice(rem_list, p=probs))
            seq.append(j)
            remaining.remove(j)
            prev = j
        return seq

    best_seq = None
    best_val = float("inf")
    hist = []

    for _ in range(n_iters):
        sols, vals = [], []
        for _a in range(n_ants):
            seq = construct_solution()
            val = total_tardiness(seq, p, d)
            sols.append(seq)
            vals.append(val)

        i_best = int(np.argmin(vals))
        if vals[i_best] < best_val:
            best_val = float(vals[i_best])
            best_seq = sols[i_best][:]

        tau *= (1.0 - rho)
        deposit = q / max(vals[i_best], 1e-9)
        prev = start
        for j in sols[i_best]:
            tau[prev, j] += deposit
            prev = j

        hist.append(best_val)

    return best_seq, best_val, hist

In [ ]:
aco_seq, aco_obj, aco_hist = aco_scheduling(p, d, n_jobs, n_ants=25, n_iters=150, alpha=1.0, beta=2.5, rho=0.25, q=2.0, seed=SEED)
aco_obj, len(aco_hist)

## 10) Summary and comparison (single run)

We compare best objective, runtime, and convergence curves.

In [ ]:
def time_run(fn):
    t0 = time.time()
    out = fn()
    t1 = time.time()
    return out, (t1 - t0)

results = {}

(out, tsec) = time_run(lambda: local_search_best_improvement(edd_seq, p, d, neighborhood="swap"))
results["Local Search"] = {"seq": out[0], "obj": out[1], "hist": out[2], "time_s": tsec}

(out, tsec) = time_run(lambda: simulated_annealing(edd_seq, p, d, n_iters=5000, T0=60.0, alpha=0.997, move_type="swap"))
results["Simulated Annealing"] = {"seq": out[0], "obj": out[1], "hist": out[2], "time_s": tsec}

(out, tsec) = time_run(lambda: tabu_search(edd_seq, p, d, n_iters=2500, tenure=12))
results["Tabu Search"] = {"seq": out[0], "obj": out[1], "hist": out[2], "time_s": tsec}

(out, tsec) = time_run(lambda: genetic_algorithm(p, d, n_jobs, pop_size=40, n_gens=250, pc=0.9, pm=0.25, elite=2))
results["Genetic Algorithm"] = {"seq": out[0], "obj": out[1], "hist": out[2], "time_s": tsec}

(out, tsec) = time_run(lambda: pso_random_keys(p, d, n_jobs, n_particles=30, n_iters=300, w=0.7, c1=1.6, c2=1.6, vmax=0.35, seed=SEED))
results["PSO (random keys)"] = {"seq": out[0], "obj": out[1], "hist": out[2], "time_s": tsec}

(out, tsec) = time_run(lambda: aco_scheduling(p, d, n_jobs, n_ants=25, n_iters=150, alpha=1.0, beta=2.5, rho=0.25, q=2.0, seed=SEED))
results["ACO"] = {"seq": out[0], "obj": out[1], "hist": out[2], "time_s": tsec}

summary = pd.DataFrame(
    [{"method": k, "best_total_tardiness": v["obj"], "runtime_s": v["time_s"], "best_sequence": v["seq"]}
     for k, v in results.items()]
).sort_values("best_total_tardiness")

summary

In [ ]:
plt.figure()
for method, r in results.items():
    plt.plot(r["hist"], label=method)
plt.xlabel("Iteration (or step)")
plt.ylabel("Best-so-far total tardiness")
plt.legend()
plt.title("Convergence comparison (single run)")
plt.show()

## 11) Extension: multi-run statistics (optional)

Use multiple random seeds for a fair comparison.

In [ ]:
def run_multiple_seeds(seeds, jobs_df):
    out_rows = []
    p_ = jobs_df.set_index("job")["p"].to_dict()
    d_ = jobs_df.set_index("job")["d"].to_dict()
    n_ = len(jobs_df)
    jobs_ = list(range(n_))
    edd_ = sorted(jobs_, key=lambda j: d_[j])

    for seed in seeds:
        random.seed(seed)
        np.random.seed(seed)

        sa_best = simulated_annealing(edd_, p_, d_, n_iters=3000, T0=60.0, alpha=0.997, move_type="swap")[1]
        ts_best = tabu_search(edd_, p_, d_, n_iters=1500, tenure=12)[1]
        ga_best = genetic_algorithm(p_, d_, n_, pop_size=30, n_gens=180, pc=0.9, pm=0.25, elite=2)[1]
        pso_best = pso_random_keys(p_, d_, n_, n_particles=25, n_iters=220, w=0.7, c1=1.6, c2=1.6, vmax=0.35, seed=seed)[1]
        aco_best = aco_scheduling(p_, d_, n_, n_ants=20, n_iters=120, alpha=1.0, beta=2.5, rho=0.25, q=2.0, seed=seed)[1]

        out_rows += [
            {"seed": seed, "method": "SA", "best_total_tardiness": sa_best},
            {"seed": seed, "method": "Tabu", "best_total_tardiness": ts_best},
            {"seed": seed, "method": "GA", "best_total_tardiness": ga_best},
            {"seed": seed, "method": "PSO", "best_total_tardiness": pso_best},
            {"seed": seed, "method": "ACO", "best_total_tardiness": aco_best},
        ]

    df = pd.DataFrame(out_rows)
    agg = df.groupby("method")["best_total_tardiness"].agg(["mean", "std", "min", "max"]).reset_index()
    return df, agg

df_runs, agg = run_multiple_seeds([0,1,2,3,4], jobs_df)
agg.sort_values("mean")

## 12) End-of-lab checklist

To extend:
- increase number of jobs in the data cell
- switch neighborhood (swap vs insert)
- tune parameters (T0, alpha, tenure, pop_size, pc, pm, PSO w/c1/c2, ACO alpha/beta/rho)
- compare multiple-run statistics